2\. Supervived fine tuining phase: The Liberatory Assistant (Modal Edition)
========================================================================

This document serves as the comprehensive technical and theoretical guide for the supervised fine-tuning (SFT) phase of the google/gemma-2b-it model. This stage represents the ideological calibration of the neural network, where base weights are specialized to prioritize de-colonial narratives, international legal frameworks (ICJ), and resistance strategies.

1\. Methodology & Theoretical Framework
---------------------------------------

The implementation strategy for this pipeline is built on three technical pillars designed to optimize training efficiency and data integrity.

### A. Resource Optimization via PEFT & LoRA)

Full parameter fine-tuning of the Gemma-2b model (approx. 2.5 billion parameters) in 16-bit precision would require over 40GB of VRAM just to load the weights, exceeding the capacity of mid-tier GPUs. By implementing Parameter-Efficient Fine-Tuning (PEFT) through Low-Rank Adaptation (LoRA), we freeze the base weights and train only small, low-rank matrices. This reduces the VRAM requirement to ~14GB, making it technically feasible to perform high-quality training on a single NVIDIA A10G GPU.

### B. Stateful Persistence in Stateless Environments

Modal’s serverless infrastructure is inherently stateless; any data stored in the container's local /workspace is destroyed once the function completes. To ensure the fine-tuned model weights (adapters) are not lost, we utilize Modal Volumes. This provides a persistent, network-attached file system that allows us to save the trained model for subsequent deployment in a UI or evaluation pipeline, solving the primary challenge of serverless data persistence.

### C. Ampere-Optimized Stability (bfloat16)

Deep learning models are highly sensitive to numerical precision during the weight-update phase. Standard float16 precision has a narrow dynamic range that often leads to "vanishing" or "exploding" gradients during fine-tuning, resulting in a failed model. We utilize the NVIDIA A10G’s Ampere Architecture, which supports bfloat16. This format provides the same dynamic range as float32 but uses 50% less memory, ensuring the training converges stably without the risk of "NaN" errors or loss divergence.

2\. Technical Implementation & Hardware Auditing
------------------------------------------------

The following sections break down the specific technical decisions made to optimize the model for the 24GB VRAM limit of the A10G GPU.

### I. Containerization & Environment Logic

*   **modal.Image**: We construct a custom Debian-slim environment. We explicitly select **Python 3.12** to match the local development environment, ensuring that serialized functions compile without version mismatches.
    
*   **.pip\_install()**: We pin bitsandbytes>=0.46.1. This is a critical requirement for "NF4" (Normal Float 4) quantization, which is the mathematical engine that allows the model to fit into memory.
    
*   **.add\_local\_file()**: To maintain security and speed, we inject only the train\_dataset.jsonl and test\_dataset.jsonl files. This prevents the cloud system from attempting to upload gigabytes of local notebook history.
    

### II. 4-Bit Quantization (The Memory Buffer)

*   **load\_in\_4bit=True**: This compresses the model's base weights from 16-bit to 4-bit. This reduces the base memory footprint from ~10GB to ~2.5GB.
    
*   **bnb\_4bit\_use\_double\_quant=True**: This further compresses the quantization constants themselves, saving an additional 0.4 bits per parameter—crucial for maintaining stability when processing long sequences (1024 tokens).
    

### III. LoRA Expressiveness Strategy

*   **LORA\_R = 32**: Most tutorials use a rank of 8 or 16. We increase this to 32 to ensure the model has the "expressive capacity" to learn the complex legal and historical nuances of the ICJ case and the BDS movement.
    
*   **target\_modules**: We target all primary projection layers (q, k, v, o, gate, up, down). This ensures the fine-tuning penetrates every stage of the model's attention mechanism, rather than just changing its surface-level vocabulary.
    

### IV. Training & Convergence Control

*   **learning\_rate=5e-5**: A conservative rate that "steers" the model towards the revolutionary dataset without causing "catastrophic forgetting" of its base logic.
    
*   **gradient\_accumulation\_steps=8**: By combining a micro-batch size of 2 with 8 accumulation steps, we simulate an effective batch size of 16. This provides a smoother gradient for the optimizer, resulting in higher-quality reasoning.
    
*   **optim="paged\_adamw\_8bit"**: This optimizer "pages" its internal state to the CPU RAM if the GPU VRAM fills up, acting as a final fail-safe against Out-of-Memory (OOM) crashes.
    

### V. Memory-Safe Evaluation Logic

*   **per\_device\_eval\_batch\_size=1**: During the evaluation phase (every 20 steps), the model must calculate the "loss" on the test set. Without this setting, the model would try to process multiple sequences at once, causing a massive memory spike that crashes the container.
    
*   **eval\_accumulation\_steps=1**: This forces the GPU to transfer evaluation metrics to the CPU immediately, keeping the VRAM clear for the next training iteration.

In [ ]:
import os
import modal

# ==========================================
# 1. MODAL CLOUD INFRASTRUCTURE SETUP
# ==========================================
# Initialize the Modal App
app = modal.App("revolutionary-finetune")

# Define the environment and explicitly inject our dataset files into the cloud image
# This replaces the deprecated 'modal.Mount' and prevents uploading unnecessary large local files
training_image = (
    modal.Image.debian_slim(python_version="3.12")
    .pip_install("torch", "trl", "peft", "bitsandbytes>=0.46.1", "transformers", "accelerate", "datasets")
    .add_local_file("train_dataset.jsonl", remote_path="/workspace/train_dataset.jsonl")
    .add_local_file("test_dataset.jsonl", remote_path="/workspace/test_dataset.jsonl")
)

# Create a persistent Volume to save the model so it isn't lost when the GPU shuts down
model_volume = modal.Volume.from_name("revolutionary-model-storage", create_if_missing=True)

# ==========================================
# 2. THE TRAINING FUNCTION (Runs on A10G)
# ==========================================
@app.function(
    image=training_image,
    gpu="a10g",                                             # Requests the NVIDIA A10 GPU
    secrets=[modal.Secret.from_name("huggingface-secret")], # Injects your HF_TOKEN securely
    volumes={"/vol": model_volume},                         # Mounts the persistent storage
    timeout=14400                                           # Allows training to run for up to 4 hours
)
def finetune_gemma():
    """This function executes entirely inside Modal's secure GPU container."""
    # Imports are placed inside the function so they load inside the Modal Image
    # Fix Memory Fragmentation warning from logs
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    import torch
    import gc
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig

    # Configuration mapped to Modal's file system
    MODEL_ID = "google/gemma-2b-it"
    DATASET_TRAIN = "/workspace/train_dataset.jsonl"
    DATASET_TEST = "/workspace/test_dataset.jsonl"
    OUTPUT_DIR = "/vol/gemma-revolutionary-assistant" # Saves to permanent cloud volume

    # Retrieve the token securely injected by the Modal wrapper
    HF_TOKEN = os.environ.get('HF_TOKEN')

    # HYPER-OPTIMIZED SETTINGS FOR NVIDIA A10 (24GB VRAM)
    LEARNING_RATE = 5e-5
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 8
    NUM_EPOCHS = 3
    LORA_R = 32
    LORA_ALPHA = 64
    MAX_SEQ_LENGTH = 1024

    def track_gpu_usage(stage=""):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            print(f"[{stage}] GPU Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")

    # 0. MEMORY CLEANUP
    gc.collect()
    torch.cuda.empty_cache()

    print(f"--- Starting Gemma-2b Fine-Tuning Pipeline on Modal (A10) ---")
    track_gpu_usage("Initial State")

    if not HF_TOKEN:
        print("Error: Hugging Face Token not found. Please ensure 'huggingface-secret' exists in Modal.")
        return

    # 1. BitsAndBytes Config (Highly optimized for A10 Ampere Architecture)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load Model
    print(f"Loading base model: {MODEL_ID}...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )
    model.config.use_cache = False

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 3. LoRA Configuration
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # 4. SFTConfig
    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,
        dataset_text_field="text",
        #max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        per_device_eval_batch_size=1,              # CRITICAL FIX: Stops the 7GB OOM spike during eval
        eval_accumulation_steps=1,                 # CRITICAL FIX: Offloads eval metrics to CPU
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="steps",
        eval_steps=20,
        bf16=True,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_steps=20,
        gradient_checkpointing=True,
        report_to="none"
    )

    # 5. Dataset Setup
    if not os.path.exists(DATASET_TRAIN):
        print(f"Error: {DATASET_TRAIN} not found. Modal could not see your local files.")
        return

    dataset = load_dataset("json", data_files={"train": DATASET_TRAIN, "test": DATASET_TEST})

    def manual_formatting_func(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            text = f"### Instruction:\n{example['instruction'][i]}\n\n### Response:\n{example['output'][i]}{tokenizer.eos_token}"
            output_texts.append(text)
        return {"text": output_texts}

    print("Pre-formatting dataset...")
    formatted_dataset = dataset.map(manual_formatting_func, batched=True)

    # 6. SFTTrainer Setup
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_dataset["train"],
        eval_dataset=formatted_dataset["test"],
        args=sft_config,
    )

    # 7. Execute Training
    print("Beginning Training (Gemma-2b Optimized for A10)...")
    try:
        trainer.train()
    except Exception as e:
        print(f"Training failed: {e}")
        track_gpu_usage("Error State")
        return

    # 8. Save
    print(f"Saving LoRA adapter to Modal Volume at {OUTPUT_DIR}...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Fine-tuning complete. Model is safely stored in your Modal Volume!")

# ==========================================
# 3. LOCAL ENTRYPOINT FOR NOTEBOOKS
# ==========================================
if __name__ == "__main__":
    # This block triggers the cloud execution when you run the notebook cell
    print("Connecting to Modal to request A10 GPU...")
    with app.run():
        finetune_gemma.remote()

Connecting to Modal to request A10 GPU...


In [ ]:
modal volume get revolutionary-model-storage /gemma-revolutionary-assistant ./my-local-revolutionary-model

Invalid syntax. Use: %modal from [env/]<app> import <function|Class>[, <function|Class> [as alias]]
